# LMA Phase 1 — Data Pipeline (Colab CPU)

Runs entirely on **Colab CPU** (no GPU needed for Phase 1). Drive is mounted at `/content/drive/MyDrive/LMA_Project`.

**Order of execution:**
1. Mount Drive + clone repo
2. **Stream A:** Clean Nepali 22 GB raw → cleaned shards (longest job, ~2–4 h)
3. **Stream B:** Download Hindi 80% from HF + clean (parallel session)
4. Clean Nepali manual shards + Hindi manual shards
5. Merge cleaned → near-dedup → splits → stats
6. Train Hindi BPE + Nepali BPE (minutes each)
7. Encode full corpora → measure tokens + manual% → final stats
8. Write report + README, commit, push

Open **two Colab sessions** for steps 2 and 3 to run in parallel.

## 0. Mount Drive + clone repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PROJ = '/content/drive/MyDrive/LMA_Project'
import os; os.makedirs(PROJ, exist_ok=True)
print('Drive mounted at', PROJ)

In [ ]:
# Clone the repo via HTTPS (Colab cannot use SSH). This is your
# public fork which has the phase-1 code (common/ + tokenizer modules).
REPO_URL = 'https://github.com/Bharadwaja04/LLM_From_Scratch.git'
REPO_DIR = '/content/individual-project-Bharadwaja04'
if not os.path.exists(REPO_DIR):
    !git clone --branch phase-1 {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull origin phase-1
import sys; sys.path.insert(0, REPO_DIR)
print('repo at', REPO_DIR)
!ls {REPO_DIR}/common/  # sanity check: should list clean.py, dedup.py, etc.

In [ ]:
import regex, datasets  # pip install regex datasets if missing
print('deps OK')

## Stream A — Nepali: clean 22 GB raw → cleaned shards
**Run this in one Colab session.** This is the longest job (~2–4 h on Colab CPU for 22 GB).
It streams the file line-by-line, so RAM stays bounded.

In [ ]:
# Clean the big downloaded file (22 GB) — single process, streaming
!cd {REPO_DIR} && python -m common.clean --lang nepali \
    --input /content/drive/MyDrive/LMA_Project/nepali/raw/downloaded/nepali_train.jsonl \
    --output-dir /content/drive/MyDrive/LMA_Project/nepali/processed/clean \
    --source-tag downloaded --shard-size 250000 --min-chars 80

In [ ]:
# Also clean the 5 GB test file
!cd {REPO_DIR} && python -m common.clean --lang nepali \
    --input /content/drive/MyDrive/LMA_Project/nepali/raw/downloaded/nepali_test.jsonl \
    --output-dir /content/drive/MyDrive/LMA_Project/nepali/processed/clean \
    --source-tag downloaded --shard-size 250000 --min-chars 80

In [ ]:
# Clean Nepali MANUAL shards (kantipur/sahityaras/cdc/national_library OCR)
import glob, subprocess, sys
manual_dirs = [
    '/content/drive/MyDrive/LMA_Project/nepali/raw/manual/extracted',
    '/content/drive/MyDrive/LMA_Project/nepali/raw/manual/sahityaras/extracted',
    '/content/drive/MyDrive/LMA_Project/nepali/raw/manual/cdc/extracted',
    '/content/drive/MyDrive/LMA_Project/nepali/raw/manual/national_library/ocr',
    '/content/drive/MyDrive/LMA_Project/nepali/raw/manual/national_library/extracted',
]
out_manual = '/content/drive/MyDrive/LMA_Project/nepali/processed/clean_manual'
for d in manual_dirs:
    for f in glob.glob(os.path.join(d, '*.jsonl')):
        print('cleaning', f)
        subprocess.run([sys.executable, '-m', 'common.clean', '--lang', 'nepali',
                        '--input', f, '--output-dir', out_manual,
                        '--source-tag', 'manual', '--min-chars', '40'], cwd=REPO_DIR)
print('Nepali manual cleaning done')

## Stream B — Hindi: download 80% from HF + clean
**Run this in a SECOND Colab session in parallel with Stream A.**

In [ ]:
# Download Hindi from HuggingFace (server-side streaming, ~4 GB target)
!cd {REPO_DIR} && python -m common.collect --lang hindi --sources wiki,mc4 \
    --output-dir /content/drive/MyDrive/LMA_Project/hindi/raw/downloaded \
    --max-chars 4000000000 --shard-size 250000

In [ ]:
# Clean Hindi downloaded shards
import glob, subprocess, sys
outd = '/content/drive/MyDrive/LMA_Project/hindi/processed/clean'
for f in sorted(glob.glob('/content/drive/MyDrive/LMA_Project/hindi/raw/downloaded/*.jsonl')):
    print('cleaning', f)
    subprocess.run([sys.executable, '-m', 'common.clean', '--lang', 'hindi',
                    '--input', f, '--output-dir', outd,
                    '--source-tag', 'downloaded', '--shard-size', '250000',
                    '--min-chars', '80'], cwd=REPO_DIR)
print('Hindi downloaded cleaning done')

In [ ]:
# Clean Hindi MANUAL (webcrawl) shards — tag as manual
import glob, subprocess, sys
outm = '/content/drive/MyDrive/LMA_Project/hindi/processed/clean_manual'
for f in sorted(glob.glob('/content/drive/MyDrive/LMA_Project/hindi/raw/manual/**/*.jsonl', recursive=True)):
    print('cleaning manual', f)
    subprocess.run([sys.executable, '-m', 'common.clean', '--lang', 'hindi',
                    '--input', f, '--output-dir', outm,
                    '--source-tag', 'manual', '--min-chars', '40'], cwd=REPO_DIR)
print('Hindi manual cleaning done')

## 4. Merge cleaned (downloaded + manual) → near-dedup → splits

In [ ]:
# Combine downloaded + manual cleaned into one dir (symlinks) then near-dedup
import os, glob, subprocess, sys
for lang in ['hindi', 'nepali']:
    merged = f'/content/drive/MyDrive/LMA_Project/{lang}/processed/merged'
    os.makedirs(merged, exist_ok=True)
    # copy (or symlink) all cleaned shards into merged/
    for src_dir in [f'/content/drive/MyDrive/LMA_Project/{lang}/processed/clean',
                    f'/content/drive/MyDrive/LMA_Project/{lang}/processed/clean_manual']:
        for f in glob.glob(os.path.join(src_dir, '*.jsonl')):
            dst = os.path.join(merged, os.path.basename(f))
            if not os.path.exists(dst):
                os.symlink(os.path.abspath(f), dst)
    print(lang, 'merged dir:', merged, '->', len(os.listdir(merged)), 'files')

In [ ]:
# Near-dedup (MinHash LSH). Skip if too slow; exact dedup already ran in clean.py.
for lang in ['hindi', 'nepali']:
    inp = f'/content/drive/MyDrive/LMA_Project/{lang}/processed/merged'
    out = f'/content/drive/MyDrive/LMA_Project/{lang}/processed/dedup'
    !cd {REPO_DIR} && python -m common.dedup --lang {lang} --input {inp} --output-dir {out} \
        --num-perm 64 --bands 8 --threshold 0.9 --min-chars 200

In [ ]:
# Splits (98/1/1, doc-level, seeded)
for lang in ['hindi', 'nepali']:
    inp = f'/content/drive/MyDrive/LMA_Project/{lang}/processed/dedup'
    out = f'/content/drive/MyDrive/LMA_Project/{lang}/splits'
    !cd {REPO_DIR} && python -m common.splits --lang {lang} --input {inp} \
        --output-dir {out} --train 0.98 --val 0.01 --test 0.01 --seed 17

## 5. Train tokenizers (BPE) — minutes each after the fix

In [ ]:
# Train Hindi BPE on the training split (subsample 80M chars)
!cd {REPO_DIR} && python -m hindi.tokenizer.train --language hindi \
    --data /content/drive/MyDrive/LMA_Project/hindi/splits/hindi_train.jsonl \
    --vocab-size 32000 --char-budget 80000000 \
    --output /content/drive/MyDrive/LMA_Project/hindi/tokenizer/hindi_tokenizer.json

In [ ]:
# Train Nepali BPE on the training split (subsample 80M chars)
!cd {REPO_DIR} && python -m nepali.tokenizer.train --language nepali \
    --data /content/drive/MyDrive/LMA_Project/nepali/splits/nepali_train.jsonl \
    --vocab-size 32000 --char-budget 80000000 \
    --output /content/drive/MyDrive/LMA_Project/nepali/tokenizer/nepali_tokenizer.json

## 6. Encode full corpora → measure tokens + manual% + stats

In [ ]:
# Corpus stats + token counts + manual/downloaded split + unknown rate
for lang in ['hindi', 'nepali']:
    splits = f'/content/drive/MyDrive/LMA_Project/{lang}/splits'
    tok = f'/content/drive/MyDrive/LMA_Project/{lang}/tokenizer/{lang}_tokenizer.json'
    heldout = f'{splits}/{lang}_heldout.jsonl'
    out = f'/content/drive/MyDrive/LMA_Project/{lang}/stats/{lang}_stats.json'
    !cd {REPO_DIR} && python -m common.stats --lang {lang} --input {splits} \
        --output {out} --tokenizer {tok} --heldout {heldout}

In [ ]:
# Display the stats
import json, glob
for lang in ['hindi', 'nepali']:
    sp = f'/content/drive/MyDrive/LMA_Project/{lang}/stats/{lang}_stats.json'
    if os.path.exists(sp):
        print(f'\n===== {lang.upper()} STATS =====')
        s = json.load(open(sp))
        for k, v in s.items():
            if k != 'source_breakdown':
                print(f'  {k}: {v}')
        print('  source_breakdown:')
        for kk, vv in s['source_breakdown'].items():
            print(f'    {kk}: {vv}')

## 7. Tokenization examples + unknown-token stats (for the report)

In [ ]:
import sys; sys.path.insert(0, REPO_DIR)
from nepali.tokenizer.tokenizer import Tokenizer as NeTok
from hindi.tokenizer.tokenizer import Tokenizer as HiTok

examples = {
  'hindi': ['नमस्ते दुनिया, यह एक परीक्षण वाक्य है।',
            'भारत एक विशाल और विविध देश है।'],
  'nepali': ['नमस्ते संसार, यो एक परीक्षण वाक्य हो।',
             'नेपाल एक सुन्दर हिमालयको देश हो।'],
}
for lang, toks in [('hindi', HiTok), ('nepali', NeTok)]:
    tpath = f'/content/drive/MyDrive/LMA_Project/{lang}/tokenizer/{lang}_tokenizer.json'
    if not os.path.exists(tpath):
        print(f'{lang}: tokenizer not found at {tpath}'); continue
    tok = toks.load(tpath)
    print(f'\n===== {lang.upper()} tokenization examples =====')
    for ex in examples[lang]:
        ids = tok.encode(ex)
        print(f'text: {ex}')
        print(f'ids : {ids}')
        print(f'dec : {tok.decode(ids)}')
        print(f'round_trip: {tok.verify_round_trip(ex)}')
        print(f'stats: {tok.tokenization_stats(ex)}')
        print()

## 8. Commit + push to phase-1 branch
Run these locally on your Mac after pulling the stats files, OR run here.

In [ ]:
# Copy stats + tokenizer JSONs into the repo (small files OK for git)
# Large data stays on Drive; README will link to Drive.
import shutil, os
for lang in ['hindi', 'nepali']:
    # tokenizer json
    src = f'/content/drive/MyDrive/LMA_Project/{lang}/tokenizer/{lang}_tokenizer.json'
    dst = f'{REPO_DIR}/{lang}/tokenizer/{lang}_tokenizer.json'
    if os.path.exists(src):
        shutil.copy(src, dst)
    # stats
    src = f'/content/drive/MyDrive/LMA_Project/{lang}/stats/{lang}_stats.json'
    dst = f'{REPO_DIR}/reports/{lang}_stats.json'
    if os.path.exists(src):
        shutil.copy(src, dst)
print('copied artifacts into repo')

In [ ]:
!cd {REPO_DIR} && git checkout phase-1 && git add -A && git status

In [ ]:
!cd {REPO_DIR} && git commit -m 'Phase 1: data pipeline, scalable BPE trainer, splits, stats, tokenizers' && git push origin phase-1